# 개별종목 조합A — LogisticRegression

## 실험 목적

KOSPI200 방향이 상승·보합·하락 중 어디인지 정해졌을 때, 같은 방향일 확률이 높은
개별종목을 찾기 위한 3분류 모델입니다.

후보는 매 거래일 **KOSPI 세부 업종지수 시가총액 상위 10개 × 업종별 KOSPI 보통주
시가총액 상위 5개**로 먼저 고정합니다. 업종의 미래 방향을 따로 예측하는 구조는 아닙니다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 원천 | HF `full/daily_price_dev.parquet`, `full/index_price_dev.parquet` |
| 홀드아웃 | `20240901` 이후 접근 금지 |
| 라벨 | T일 판단 → T+1 `adj_open` 진입 → T+6 `adj_open` 평가, 종목 ±2% |
| 외부 검증 | 날짜 그룹 expanding 12폴드 |
| 최초 학습 | 750거래일 |
| 검증·gap | 폴드당 60거래일 · 직전 5거래일 제거 |
| class weight | 각 외부 폴드 내부에서 `None`과 `balanced` 재비교 |
| 선정 지표 | Accuracy·Macro F1·하락 Recall 조화평균 |

## OOS 결과

| Accuracy | Macro F1 | 하락 Recall | 핵심지표 조화평균 |
|---:|---:|---:|---:|
| 0.4012 | 0.3556 | 0.2627 | **0.3115** |

아래 셀은 저장된 실측 리포트에서 이 모델의 폴드 결과와 class weight 선택 횟수를 다시
읽습니다. 학습 구현은 `models/stock_experiment.py`, 피처·라벨은
`features/stock_model_dataset.py`가 정본입니다.


In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "reports" / "stock_feature_combinations.json").exists():
    ROOT = ROOT.parent
report_path = ROOT / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination = 'A'
model_name = 'LogisticRegression'

combination_report = report["combinations"][combination]
folds = pd.DataFrame(combination_report["outer_fold_results"])
display(folds.loc[folds["model"].eq(model_name)].reset_index(drop=True))

weights = pd.DataFrame(combination_report["selected_class_weight_counts"])
display(weights.loc[weights["model"].eq(model_name)].reset_index(drop=True))


,model,fold,selected_class_weight,train_dates,valid_dates,train_rows,valid_rows,train_end,valid_start,valid_end,...,down_recall,core_harmonic_mean,training_majority_class,training_majority_baseline_accuracy,validation_majority_class,validation_majority_oracle_accuracy,validation_down_rate,validation_neutral_rate,validation_up_rate,accuracy_minus_training_majority_baseline
0,LogisticRegression,1,balanced,750,60,36207,2880,20130402,20130410,20130705,...,0.159018,0.253929,0,0.370139,0,0.370139,0.325347,0.370139,0.304514,0.017361
1,LogisticRegression,2,balanced,999,60,48107,2880,20140404,20140414,20140711,...,0.106040,0.201404,0,0.474306,0,0.474306,0.258681,0.474306,0.267014,-0.007292
2,LogisticRegression,3,balanced,1248,60,60029,2884,20150413,20150421,20150716,...,0.344796,0.359213,0,0.330097,-1,0.373093,0.373093,0.330097,0.296810,0.036755
3,LogisticRegression,4,balanced,1496,60,71990,2973,20160414,20160422,20160719,...,0.230692,0.322211,0,0.410696,0,0.410696,0.335351,0.410696,0.253952,0.015473
4,LogisticRegression,5,balanced,1745,60,84070,2872,20170414,20170424,20170721,...,0.182779,0.274886,0,0.417827,0,0.417827,0.230501,0.417827,0.351671,0.003830
5,LogisticRegression,6,balanced,1994,60,95927,2936,20180424,20180503,20180731,...,0.310865,0.357857,0,0.390668,0,0.390668,0.338556,0.390668,0.270777,-0.000681
6,LogisticRegression,7,balanced,2243,60,108084,2940,20190503,20190514,20190806,...,0.095797,0.190364,0,0.461224,0,0.461224,0.347959,0.461224,0.190816,-0.032653
7,LogisticRegression,8,balanced,2492,60,120258,2936,20200508,20200518,20200807,...,0.568493,0.408430,0,0.314373,1,0.387262,0.298365,0.314373,0.387262,0.047684
8,LogisticRegression,9,balanced,2741,60,132456,2956,20210510,20210518,20210810,...,0.327815,0.381397,0,0.441813,0,0.441813,0.306495,0.441813,0.251691,0.000338
9,LogisticRegression,10,balanced,2989,60,144792,3000,20220511,20220519,20220812,...,0.216667,0.291990,0,0.334333,-1,0.340000,0.340000,0.334333,0.325667,0.022667


,model,selected_class_weight,folds
0,LogisticRegression,balanced,12
